Step 1: Setup Environment

In [1]:
# Init Minio
!sh /home/LocalLakeHouse/Project/Scripts/1.init_minio.sh "data"


Data subdir: data
Source path: /home/LocalLakeHouse/Project/data
Platform: arm64

mc already exists

Check if the bucket already exists...
]11;?\[2026-04-24 04:02:12 UTC]     0B Bronze/
[2026-04-24 04:02:12 UTC]     0B Gold/
[2026-04-24 04:02:12 UTC]     0B Raw/
[2026-04-24 04:02:12 UTC]     0B Silver/
Bucket already exists

Uploading data from '/home/LocalLakeHouse/Project/data' to 'minio/data-lakehouse/Raw'

...lation.csv: 240.69 MiB / 240.69 MiB ┃▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓▓┃ 109.72 MiB/s 2s

In [2]:
# Install necessary packages
import sys
!{sys.executable} -m pip install pyspark
!{sys.executable} -m pip install s3fs
!{sys.executable} -m pip install minio
!{sys.executable} -m pip install pyhive
!{sys.executable} -m pip install trino

In [3]:
# Install dotenv to load environment variables
!{sys.executable} -m pip install python-dotenv

In [4]:
# Load environment variables
import os
from dotenv import load_dotenv
load_dotenv('minio.env')

# Access the environment variables
minio_access_key = os.getenv('MINIO_ACCESS_KEY')
minio_secret_key = os.getenv('MINIO_SECRET_KEY')
minio_endpoint = os.getenv('MINIO_ENDPOINT', "http://minio:9000")
minio_bucket_name = os.getenv('MINIO_BUCKET_NAME', "data-lakehouse")

In [5]:
print("Minio Access Key:", minio_access_key)
print("Minio Secret Key:", minio_secret_key)
print("Minio Endpoint:", minio_endpoint)
print("Minio Bucket Name:", minio_bucket_name)

Minio Access Key: minio_ak
Minio Secret Key: minio_sk
Minio Endpoint: http://minio:9000
Minio Bucket Name: data-lakehouse


In [ ]:
# Import Spark libraries
import pyspark
import pyspark.sql.functions as sqlF
from pyspark.sql import SparkSessionf

In [12]:
# Import SLAlchemy and Pandas libraries
from sqlalchemy.sql import text
from sqlalchemy import create_engine
import pandas as pd

In [8]:
# Initialize Spark session
spark = SparkSession.builder \
    .appName("OlistDataIngestion") \
    .config("spark.driver.host", "localhost") \
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.1.0,org.apache.hadoop:hadoop-aws:3.3.3") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("hive.metastore.uris", "thrift://metastore:9083") \
    .enableHiveSupport() \
    .getOrCreate()

bash: warning: setlocale: LC_ALL: cannot change locale (en_US.UTF-8)
bash: warning: setlocale: LC_ALL: cannot change locale (en_US.UTF-8)
Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-f2c8654d-2f02-4bf5-aae5-2ec56c52b055;1.0
	confs: [default]


:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found io.delta#delta-spark_2.12;3.1.0 in central
	found io.delta#delta-storage;3.1.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.hadoop#hadoop-aws;3.3.3 in central
	found com.amazonaws#aws-java-sdk-bundle;1.11.1026 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 188ms :: artifacts dl 6ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.11.1026 from central in [default]
	io.delta#delta-spark_2.12;3.1.0 from central in [default]
	io.delta#delta-storage;3.1.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.3 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	

In [9]:
# Check spark configuration
spark.sparkContext.getConf().getAll()

[('spark.eventLog.enabled', 'true'),
 ('spark.hadoop.fs.s3a.connection.ssl.enabled', 'false'),
 ('spark.repl.local.jars',
  'file:///root/.ivy2/jars/io.delta_delta-spark_2.12-3.1.0.jar,file:///root/.ivy2/jars/org.apache.hadoop_hadoop-aws-3.3.3.jar,file:///root/.ivy2/jars/io.delta_delta-storage-3.1.0.jar,file:///root/.ivy2/jars/org.antlr_antlr4-runtime-4.9.3.jar,file:///root/.ivy2/jars/com.amazonaws_aws-java-sdk-bundle-1.11.1026.jar,file:///root/.ivy2/jars/org.wildfly.openssl_wildfly-openssl-1.0.7.Final.jar'),
 ('spark.hadoop.fs.s3a.aws.credentials.provider',
  'org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider'),
 ('spark.hadoop.fs.s3a.imp', 'org.apache.hadoop.fs.s3a.S3AFileSystem'),
 ('spark.sql.parquet.int96RebaseModeInWrite', 'CORRECTED'),
 ('spark.serializer', 'org.apache.spark.serializer.KryoSerializer'),
 ('spark.hadoop.fs.s3a.path.style.access', 'True'),
 ('spark.files',
  'file:///root/.ivy2/jars/io.delta_delta-spark_2.12-3.1.0.jar,file:///root/.ivy2/jars/org.apache.hadoop_

Step 2: Build the data frames

In [13]:
df_test = spark.read.csv("s3a://data-lakehouse/Raw/data/olist/olist_orders_dataset.csv", header=True)
df_test.show(5)

+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|  order_approved_at|order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------+--------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|e481f51cbdc54678b...|9ef432eb625129730...|   delivered|     2017-10-02 10:56:33|2017-10-02 11:07:15|         2017-10-04 19:55:00|          2017-10-10 21:25:13|          2017-10-18 00:00:00|
|53cdb2fc8bc7dce0b...|b0830fb4747a6c6d2...|   delivered|     2018-07-24 20:41:37|2018-07-26 03:24:27|         2018-07-26 14:31:00|          2018-08-07 15:27:45|          2018-08-13 00:00:00|
|47770eb9100c2d0c4...|41ce2a54c0b03bf34...|  

In [14]:
from pyspark.sql.functions import current_timestamp, input_file_name

In [18]:
def ingest_to_bronze(table_name):
    # Đường dẫn nguồn (Raw) và đích (Bronze)
    raw_path = f"s3a://data-lakehouse/Raw/data/olist/{table_name}.csv"
    bronze_path = f"s3a://data-lakehouse/Bronze/{table_name}"
    
    print(f"Đang xử lý: {table_name}...")
    
    # 1. Đọc từ Raw (giữ nguyên schema gốc)
    df_raw = spark.read.csv(raw_path, header=True, inferSchema=True)
    
    # 2. Thêm cột Metadata (Chuẩn Bronze)
    df_bronze = df_raw.withColumn("ingestion_at", current_timestamp()) \
                      .withColumn("source_file", input_file_name())
    
    # 3. Ghi xuống Bronze dưới dạng Delta
    df_bronze.write.format("delta") \
             .mode("overwrite") \
             .option("overwriteSchema", "true") \
             .save(bronze_path)
    
    print(f"Đã hoàn thành nạp vào Bronze: {bronze_path}")

In [16]:
olist_tables = [
    "olist_customers_dataset",
    "olist_geolocation_dataset",
    "olist_order_items_dataset",
    "olist_order_payments_dataset",
    "olist_order_reviews_dataset",
    "olist_orders_dataset",
    "olist_products_dataset",
    "olist_sellers_dataset",
    "product_category_name_translation"
]

In [20]:
for table in olist_tables:
    try:
        ingest_to_bronze(table)
    except Exception as e:
        print(f"Lỗi khi nạp bảng {table}: {str(e)}")

Đang xử lý: olist_customers_dataset...
Đã hoàn thành nạp vào Bronze: s3a://data-lakehouse/Bronze/olist_customers_dataset
Đang xử lý: olist_geolocation_dataset...


26/04/24 04:53:36 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


Đã hoàn thành nạp vào Bronze: s3a://data-lakehouse/Bronze/olist_geolocation_dataset
Đang xử lý: olist_order_items_dataset...


Đã hoàn thành nạp vào Bronze: s3a://data-lakehouse/Bronze/olist_order_items_dataset
Đang xử lý: olist_order_payments_dataset...
Đã hoàn thành nạp vào Bronze: s3a://data-lakehouse/Bronze/olist_order_payments_dataset
Đang xử lý: olist_order_reviews_dataset...


Đã hoàn thành nạp vào Bronze: s3a://data-lakehouse/Bronze/olist_order_reviews_dataset
Đang xử lý: olist_orders_dataset...


Đã hoàn thành nạp vào Bronze: s3a://data-lakehouse/Bronze/olist_orders_dataset
Đang xử lý: olist_products_dataset...
Đã hoàn thành nạp vào Bronze: s3a://data-lakehouse/Bronze/olist_products_dataset
Đang xử lý: olist_sellers_dataset...
Đã hoàn thành nạp vào Bronze: s3a://data-lakehouse/Bronze/olist_sellers_dataset
Đang xử lý: product_category_name_translation...
Đã hoàn thành nạp vào Bronze: s3a://data-lakehouse/Bronze/product_category_name_translation


In [21]:
df_orders_bronze = spark.read.format("delta").load(f"s3a://data-lakehouse/Bronze/olist_orders_dataset")

# Hiển thị 5 dòng đầu kèm các cột metadata mới
df_orders_bronze.select("order_id", "order_status", "ingestion_at", "source_file").show(5, False)

# Đếm tổng số dòng để đảm bảo không mất mát dữ liệu
print(f"Tổng số dòng trong lớp Bronze: {df_orders_bronze.count()}")

+--------------------------------+------------+--------------------------+------------------------------------------------------------+
|order_id                        |order_status|ingestion_at              |source_file                                                 |
+--------------------------------+------------+--------------------------+------------------------------------------------------------+
|995392413cee61cc1fcec9807870ea8c|delivered   |2026-04-24 04:53:46.393605|s3a://data-lakehouse/Raw/data/olist/olist_orders_dataset.csv|
|b39de9ed2bb8fd08a970e4517c698824|delivered   |2026-04-24 04:53:46.393605|s3a://data-lakehouse/Raw/data/olist/olist_orders_dataset.csv|
|b1a88554eb1f7f68627549e26257b583|delivered   |2026-04-24 04:53:46.393605|s3a://data-lakehouse/Raw/data/olist/olist_orders_dataset.csv|
|fe2ac30768e07e362b649951cc625144|delivered   |2026-04-24 04:53:46.393605|s3a://data-lakehouse/Raw/data/olist/olist_orders_dataset.csv|
|460673777918e7a42a262293341ae7d5|delivered   |2